<a href="https://colab.research.google.com/github/Atikah-264/sql-mini-analysis/blob/main/hands_on_sesi_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# set up google collab

In [ ]:
!pip -q install google-cloud-bigquery db-dtypes openpyxl


In [ ]:
from google.colab import auth

auth.authenticate_user()

print("Google authentication berhasil.")


Google authentication berhasil.


In [ ]:
# connect to big query

from google.cloud import bigquery

PROJECT_ID = "atikahs-project"

client = bigquery.Client(project=PROJECT_ID)

print("Connected to:", PROJECT_ID)

Connected to: atikahs-project


In [ ]:
query = '''
SELECT
  table_name
FROM `bigquery-public-data.thelook_ecommerce.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
'''

client.query(query).to_dataframe()

,table_name
0,distribution_centers
1,events
2,inventory_items
3,order_items
4,orders
5,products
6,users


Sebelum JOIN, kita lihat kolom penting yang tersedia.

In [ ]:
query = '''
SELECT
  table_name,
  column_name,
  data_type
FROM `bigquery-public-data.thelook_ecommerce.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN ('orders', 'users', 'order_items', 'products')
ORDER BY table_name, ordinal_position
'''

schema = client.query(query).to_dataframe()
schema


,table_name,column_name,data_type
0,order_items,id,INT64
1,order_items,order_id,INT64
2,order_items,user_id,INT64
3,order_items,product_id,INT64
4,order_items,inventory_item_id,INT64
5,order_items,status,STRING
6,order_items,created_at,TIMESTAMP
7,order_items,shipped_at,TIMESTAMP
8,order_items,delivered_at,TIMESTAMP
9,order_items,returned_at,TIMESTAMP


 SQL FINAL — SALES DATASET

Kita menggabungkan Orders, Users,Order Items, Products,
Outputnya berada pada level order item.

Artinya satu order dapat memiliki beberapa baris produk.

In [ ]:
query = '''
SELECT
  o.order_id,
  o.user_id,

  -- Order information
  o.status AS order_status,
  o.created_at AS order_created_at,

  -- Customer information
  u.gender,
  u.age,
  u.country,

  -- Product transaction information
  oi.id AS order_item_id,
  oi.product_id,
  oi.status AS item_status,
  oi.created_at AS item_created_at,
  oi.sale_price,

  -- Product information
  p.name AS product_name,
  p.category,
  p.department,
  p.brand

FROM `bigquery-public-data.thelook_ecommerce.orders` AS o

LEFT JOIN `bigquery-public-data.thelook_ecommerce.users` AS u
  ON o.user_id = u.id

LEFT JOIN `bigquery-public-data.thelook_ecommerce.order_items` AS oi
  ON o.order_id = oi.order_id

LEFT JOIN `bigquery-public-data.thelook_ecommerce.products` AS p
  ON oi.product_id = p.id
'''

sales_df = client.query(query).to_dataframe()

print("Rows   :", sales_df.shape[0])
print("Columns:", sales_df.shape[1])


Rows   : 182295
Columns: 16


In [ ]:
# lihat hasil join

sales_df.head()

,order_id,user_id,order_status,order_created_at,gender,age,country,order_item_id,product_id,item_status,item_created_at,sale_price,product_name,category,department,brand
0,24,18,Cancelled,2026-05-12 21:28:19+00:00,F,48,United States,36,4128,Cancelled,2026-05-12 17:36:08+00:00,48.75,Vintage Havana Womens Stripe Romper,Jumpsuits & Rompers,Women,Vintage
1,34,25,Cancelled,2026-09-26 03:05:47.934217+00:00,F,22,China,52,6536,Cancelled,2026-09-27 00:08:04.934217+00:00,7.44,Allegra K Ladies Elastic Waist Self Bowknot Fr...,Shorts,Women,Allegra K
2,34,25,Cancelled,2026-09-26 03:05:47.934217+00:00,F,22,China,51,13567,Cancelled,2026-09-29 23:41:55.934217+00:00,31.92,Stohlquist Women's Short Sleeve Burnout Rashguard,Swim,Women,Stohlquist
3,34,25,Cancelled,2026-09-26 03:05:47.934217+00:00,F,22,China,50,2206,Cancelled,2026-09-29 23:53:36.934217+00:00,50.00,Live for the Ride Paisley Pony Hoodie,Fashion Hoodies & Sweatshirts,Women,Live for the Ride
4,34,25,Cancelled,2026-09-26 03:05:47.934217+00:00,F,22,China,53,8507,Cancelled,2026-09-27 02:48:43.934217+00:00,187.50,MICHAEL Michael Kors Women's Zip Jacket,Outerwear & Coats,Women,MICHAEL Michael Kors


In [ ]:
# check column dataset final

sales_df.columns.tolist()

['order_id',
 'user_id',
 'order_status',
 'order_created_at',
 'gender',
 'age',
 'country',
 'order_item_id',
 'product_id',
 'item_status',
 'item_created_at',
 'sale_price',
 'product_name',
 'category',
 'department',
 'brand']

In [ ]:
# check data type

sales_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 182295 entries, 0 to 182294
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype              
---  ------            --------------   -----              
 0   order_id          182295 non-null  Int64              
 1   user_id           182295 non-null  Int64              
 2   order_status      182295 non-null  object             
 3   order_created_at  182295 non-null  datetime64[us, UTC]
 4   gender            182295 non-null  object             
 5   age               182295 non-null  Int64              
 6   country           182295 non-null  object             
 7   order_item_id     182295 non-null  Int64              
 8   product_id        182295 non-null  Int64              
 9   item_status       182295 non-null  object             
 10  item_created_at   182295 non-null  datetime64[us, UTC]
 11  sale_price        182295 non-null  float64            
 12  product_name      182276 non-null  object   

In [ ]:
# check duplicate

duplicate_order_items = sales_df["order_item_id"].duplicated().sum()

print("Duplicate order_item_id:", duplicate_order_items)


Duplicate order_item_id: 0


In [ ]:
# celeaning date

import pandas as pd

sales_df["order_created_at"] = pd.to_datetime(
    sales_df["order_created_at"],
    errors="coerce"
)

sales_df["order_date"] = (
    sales_df["order_created_at"].dt.date
)

sales_df["year"] = (
    sales_df["order_created_at"].dt.year
)

sales_df["month"] = (
    sales_df["order_created_at"].dt.month
)

sales_df["month_name"] = (
    sales_df["order_created_at"].dt.strftime("%B")
)

sales_df["year_month"] = (
    sales_df["order_created_at"]
    .dt.to_period("M")
    .astype(str)
)

sales_df.head()

/tmp/ipykernel_1029/678404948.py:28: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


,order_id,user_id,order_status,order_created_at,gender,age,country,order_item_id,product_id,item_status,...,sale_price,product_name,category,department,brand,order_date,year,month,month_name,year_month
0,24,18,Cancelled,2026-05-12 21:28:19+00:00,F,48,United States,36,4128,Cancelled,...,48.75,Vintage Havana Womens Stripe Romper,Jumpsuits & Rompers,Women,Vintage,2026-05-12,2026,5,May,2026-05
1,34,25,Cancelled,2026-09-26 03:05:47.934217+00:00,F,22,China,52,6536,Cancelled,...,7.44,Allegra K Ladies Elastic Waist Self Bowknot Fr...,Shorts,Women,Allegra K,2026-09-26,2026,9,September,2026-09
2,34,25,Cancelled,2026-09-26 03:05:47.934217+00:00,F,22,China,51,13567,Cancelled,...,31.92,Stohlquist Women's Short Sleeve Burnout Rashguard,Swim,Women,Stohlquist,2026-09-26,2026,9,September,2026-09
3,34,25,Cancelled,2026-09-26 03:05:47.934217+00:00,F,22,China,50,2206,Cancelled,...,50.00,Live for the Ride Paisley Pony Hoodie,Fashion Hoodies & Sweatshirts,Women,Live for the Ride,2026-09-26,2026,9,September,2026-09
4,34,25,Cancelled,2026-09-26 03:05:47.934217+00:00,F,22,China,53,8507,Cancelled,...,187.50,MICHAEL Michael Kors Women's Zip Jacket,Outerwear & Coats,Women,MICHAEL Michael Kors,2026-09-26,2026,9,September,2026-09


In [ ]:
# make sales column

sales_df["sales_amount"] = sales_df["sale_price"]

sales_df[
    [
        "order_id",
        "order_item_id",
        "product_name",
        "quantity" if "quantity" in sales_df.columns else "sale_price",
        "sales_amount"
    ]
].head()


,order_id,order_item_id,product_name,sale_price,sales_amount
0,24,36,Vintage Havana Womens Stripe Romper,48.75,48.75
1,34,52,Allegra K Ladies Elastic Waist Self Bowknot Fr...,7.44,7.44
2,34,51,Stohlquist Women's Short Sleeve Burnout Rashguard,31.92,31.92
3,34,50,Live for the Ride Paisley Pony Hoodie,50.00,50.00
4,34,53,MICHAEL Michael Kors Women's Zip Jacket,187.50,187.50


In [ ]:
# kpi validilasi

total_sales = sales_df["sales_amount"].sum()

total_order_items = sales_df["order_item_id"].nunique()

total_orders = sales_df["order_id"].nunique()

print("Total Sales       :", round(total_sales, 2))
print("Total Order Items :", total_order_items)
print("Total Orders      :", total_orders)


Total Sales       : 10865070.49
Total Order Items : 182295
Total Orders      : 125588


In [ ]:
# sales by country

sales_by_country = (
    sales_df
    .groupby("country", dropna=False)["sales_amount"]
    .sum()
    .reset_index()
    .sort_values("sales_amount", ascending=False)
)

sales_by_country.head(10)


,country,sales_amount
4,China,3.686020e+06
14,United States,2.468557e+06
3,Brasil,1.587194e+06
11,South Korea,5.823345e+05
7,France,5.030963e+05
13,United Kingdom,4.821607e+05
8,Germany,4.620634e+05
12,Spain,4.317934e+05
9,Japan,2.616715e+05
0,Australia,2.418609e+05


In [ ]:
# sales by category

sales_by_category = (
    sales_df
    .groupby("category", dropna=False)["sales_amount"]
    .sum()
    .reset_index()
    .sort_values("sales_amount", ascending=False)
)

sales_by_category.head(10)


,category,sales_amount
11,Outerwear & Coats,1.329200e+06
7,Jeans,1.261629e+06
22,Sweaters,8.411076e+05
21,Suits & Sport Coats,6.645918e+05
23,Swim,6.555810e+05
5,Fashion Hoodies & Sweatshirts,6.414170e+05
17,Sleep & Lounge,5.461996e+05
15,Shorts,5.203914e+05
24,Tops & Tees,4.990618e+05
4,Dresses,4.571993e+05


In [ ]:
# sales by month

sales_by_month = (
    sales_df
    .groupby("year_month")["sales_amount"]
    .sum()
    .reset_index()
    .sort_values("year_month")
)

sales_by_month.head(10)

,year_month,sales_amount
0,2019-01,472.980001
1,2019-02,1985.870015
2,2019-03,4753.580009
3,2019-04,3445.690008
4,2019-05,4782.910013
5,2019-06,7168.819994
6,2019-07,8096.190007
7,2019-08,11155.540005
8,2019-09,12509.919992
9,2019-10,10212.790013


In [ ]:
# sales by status

sales_by_status = (
    sales_df
    .groupby("order_status", dropna=False)["sales_amount"]
    .sum()
    .reset_index()
    .sort_values("sales_amount", ascending=False)
)

sales_by_status


,order_status,sales_amount
4,Shipped,3.260135e+06
1,Complete,2.732343e+06
2,Processing,2.153779e+06
0,Cancelled,1.634573e+06
3,Returned,1.084241e+06


In [ ]:
# sales bby gender

sales_by_gender = (
    sales_df
    .groupby("gender", dropna=False)["sales_amount"]
    .sum()
    .reset_index()
    .sort_values("sales_amount", ascending=False)
)

sales_by_gender


,gender,sales_amount
1,M,5.796459e+06
0,F,5.068611e+06


In [ ]:
# top 10 product

top_products = (
    sales_df
    .groupby(
        ["product_id", "product_name"],
        dropna=False
    )["sales_amount"]
    .sum()
    .reset_index()
    .sort_values("sales_amount", ascending=False)
    .head(10)
)

top_products


,product_id,product_name,sales_amount
24364,24428,The North Face Apex Bionic Mens Soft Shell Ski...,13545.0
22750,22812,Quiksilver Men's Rockefeller Walkshort,10836.0
18600,18656,JORDAN DURASHEEN SHORT MENS 404309-109,9933.0
23584,23646,Diesel Men's Lophophora Leather Jacket,9878.0
23740,23803,Canada Goose Men's The Chateau Jacket,9780.0
23925,23989,The North Face Freedom Mens Ski Pants 2013,9030.0
24046,24110,Woolrich Arctic Parka DF,8910.0
24277,24341,Nobis Yatesy Parka,8550.0
24219,24283,The North Face Nuptse 2 Jacket - Noah Green/TN...,8127.0
23679,23742,Canada Goose Men's Citadel Parka,7950.0


# final data quality check

In [ ]:
print("===== FINAL DATA QUALITY CHECK =====")
print("Rows             :", len(sales_df))
print("Columns          :", len(sales_df.columns))
print("Missing values   :", int(sales_df.isna().sum().sum()))
print("Duplicate items  :", int(sales_df["order_item_id"].duplicated().sum()))
print("Total Orders     :", sales_df["order_id"].nunique())
print("Total Items      :", sales_df["order_item_id"].nunique())
print("Total Sales      :", round(sales_df["sales_amount"].sum(), 2))


===== FINAL DATA QUALITY CHECK =====
Rows             : 182295
Columns          : 22
Missing values   : 174
Duplicate items  : 0
Total Orders     : 125588
Total Items      : 182295
Total Sales      : 10865070.49


# final column untuk excel

In [ ]:
final_columns = [
    "order_id",
    "order_item_id",
    "user_id",
    "order_status",
    "item_status",
    "order_created_at",
    "order_date",
    "year",
    "month",
    "month_name",
    "year_month",
    "gender",
    "age",
    "country",
    "product_id",
    "product_name",
    "category",
    "department",
    "brand",
    "sale_price",
    "sales_amount"
]

final_sales_df = sales_df[final_columns].copy()

final_sales_df.head()


,order_id,order_item_id,user_id,order_status,item_status,order_created_at,order_date,year,month,month_name,...,gender,age,country,product_id,product_name,category,department,brand,sale_price,sales_amount
0,24,36,18,Cancelled,Cancelled,2026-05-12 21:28:19+00:00,2026-05-12,2026,5,May,...,F,48,United States,4128,Vintage Havana Womens Stripe Romper,Jumpsuits & Rompers,Women,Vintage,48.75,48.75
1,34,52,25,Cancelled,Cancelled,2026-09-26 03:05:47.934217+00:00,2026-09-26,2026,9,September,...,F,22,China,6536,Allegra K Ladies Elastic Waist Self Bowknot Fr...,Shorts,Women,Allegra K,7.44,7.44
2,34,51,25,Cancelled,Cancelled,2026-09-26 03:05:47.934217+00:00,2026-09-26,2026,9,September,...,F,22,China,13567,Stohlquist Women's Short Sleeve Burnout Rashguard,Swim,Women,Stohlquist,31.92,31.92
3,34,50,25,Cancelled,Cancelled,2026-09-26 03:05:47.934217+00:00,2026-09-26,2026,9,September,...,F,22,China,2206,Live for the Ride Paisley Pony Hoodie,Fashion Hoodies & Sweatshirts,Women,Live for the Ride,50.00,50.00
4,34,53,25,Cancelled,Cancelled,2026-09-26 03:05:47.934217+00:00,2026-09-26,2026,9,September,...,F,22,China,8507,MICHAEL Michael Kors Women's Zip Jacket,Outerwear & Coats,Women,MICHAEL Michael Kors,187.50,187.50


In [ ]:
# MENGHILANGKAN TIMEZONE
final_sales_df['order_created_at'] = final_sales_df['order_created_at'].dt.tz_convert(None)

# data untuk excel

In [ ]:
excel_file = "TheLook_Sales_Dashboard_Dataset.xlsx"

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    final_sales_df.to_excel(
        writer,
        sheet_name="sales_data",
        index=False
    )

    # Reference sheets untuk validasi peserta
    sales_by_country.to_excel(
        writer,
        sheet_name="ref_country",
        index=False
    )

    sales_by_category.to_excel(
        writer,
        sheet_name="ref_category",
        index=False
    )

    sales_by_month.to_excel(
        writer,
        sheet_name="ref_month",
        index=False
    )

    sales_by_status.to_excel(
        writer,
        sheet_name="ref_status",
        index=False
    )

    top_products.to_excel(
        writer,
        sheet_name="ref_product",
        index=False
    )

print("Excel berhasil dibuat:", excel_file)

Excel berhasil dibuat: TheLook_Sales_Dashboard_Dataset.xlsx


In [ ]:
# DOWNLOAD FILE

from google.colab import files

files.download(
    "TheLook_Sales_Dashboard_Dataset.xlsx"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>